In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nlp-ass2/train.json
/kaggle/input/nlp-ass2/val.json


In [10]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import re
import pickle
import gensim.downloader as gensim_downloader
from collections import Counter
import logging
import sys
import zipfile
import requests
from io import BytesIO

# Setting up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")

# Download conlleval.py if not exists
def download_conlleval():
    if not os.path.exists('conlleval.py'):
        logger.info("Downloading conlleval.py for evaluation...")
        url = "https://raw.githubusercontent.com/sighsmile/conlleval/master/conlleval.py"
        response = requests.get(url)
        with open('conlleval.py', 'wb') as f:
            f.write(response.content)
        logger.info("Downloaded conlleval.py")
    else:
        logger.info("conlleval.py already exists")

# Download embeddings if not exists
def download_embeddings():
    if not os.path.exists('embeddings'):
        os.makedirs('embeddings')
        
    # Download GloVe embeddings
    if not os.path.exists('embeddings/glove.6B.100d.txt'):
        logger.info("Downloading GloVe embeddings...")
        url = "http://nlp.stanford.edu/data/glove.6B.zip"
        response = requests.get(url)
        with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
            zip_ref.extractall('embeddings')
        logger.info("Downloaded GloVe embeddings")

    # Download FastText embeddings
    if not os.path.exists('embeddings/wiki-news-300d-1M.vec'):
        logger.info("Downloading FastText embeddings...")
        url = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip"
        response = requests.get(url)
        with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
            zip_ref.extractall('embeddings')
        logger.info("Downloaded FastText embeddings")

# Import conlleval for F1 score calculation
try:
    from conlleval import evaluate
except ImportError:
    download_conlleval()
    from conlleval import evaluate

# Constants
GLOVE_PATH = '/kaggle/input/glove/glove.6B.100d.txt'
FASTTEXT_PATH = '/kaggle/input/fasttext/wiki-news-300d-1M.vec'
TRAIN_PATH = '/kaggle/input/nlp-ass2/train.json'
VAL_PATH = '/kaggle/input/nlp-ass2/val.json'
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
HIDDEN_DIM = 128
EMBEDDING_DIM_GLOVE = 100
EMBEDDING_DIM_FASTTEXT = 300
MAX_LEN = 100

# Check if files exist
if not os.path.exists(TRAIN_PATH) or not os.path.exists(VAL_PATH):
    logger.error(f"Data files not found at: {TRAIN_PATH} or {VAL_PATH}")
    sys.exit(1)

# Check if embeddings exist
if not os.path.exists(GLOVE_PATH):
    logger.warning(f"GloVe embeddings not found at: {GLOVE_PATH}. Will use downloaded version.")
    GLOVE_PATH = 'embeddings/glove.6B.100d.txt'
    download_embeddings()

if not os.path.exists(FASTTEXT_PATH):
    logger.warning(f"FastText embeddings not found at: {FASTTEXT_PATH}. Will use downloaded version.")
    FASTTEXT_PATH = 'embeddings/wiki-news-300d-1M.vec'
    download_embeddings()

In [11]:
def evaluate_model(model, data_loader, criterion, label_vocab):
    model.eval()
    total_loss = 0
    all_predictions = []
    all_targets = []
    conll_eval_lines = []  # To store lines for conlleval

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            tokens = batch['tokens'].to(device)
            labels = batch['labels'].to(device)
            lengths = batch['lengths']

            # Forward pass
            logits = model(tokens, lengths)

            # Reshape for loss calculation
            batch_size, max_len, num_classes = logits.shape
            logits_flat = logits.view(-1, num_classes)
            labels_flat = labels.view(-1)

            # Mask padding
            mask = (labels_flat != 0).float()

            # Compute loss
            loss = criterion(logits_flat, labels_flat) * mask
            loss = loss.sum() / mask.sum()
            total_loss += loss.item()

            # Get predictions
            _, predictions = torch.max(logits, dim=2)

            # Convert predictions and targets to lists
            for i in range(batch_size):
                length = lengths[i].item()
                pred = predictions[i, :length].cpu().tolist()
                target = labels[i, :length].cpu().tolist()
                token_indices = tokens[i, :length].cpu().tolist()

                # Convert indices to BIO labels
                pred_labels = [label_vocab.idx_to_token(p) for p in pred]
                true_labels = [label_vocab.idx_to_token(t) for t in target]
                tokens_text = [word_vocab.idx_to_token(t) for t in token_indices]

                # Append to global lists
                all_predictions.extend(pred_labels)
                all_targets.extend(true_labels)

                # Create lines for conlleval
                for token, true_label, pred_label in zip(tokens_text, true_labels, pred_labels):
                    if token != word_vocab.pad_token:  # Skip padding tokens
                        conll_eval_lines.append(f"{token} {true_label} {pred_label}")
                conll_eval_lines.append("")  # Blank line to separate sentences

    # Calculate F1 score (ignoring padding)
    non_pad_indices = [i for i, t in enumerate(all_targets) if t != 0]
    non_pad_predictions = [all_predictions[i] for i in non_pad_indices]
    non_pad_targets = [all_targets[i] for i in non_pad_indices]

    # Regular F1 score
    f1 = f1_score(non_pad_targets, non_pad_predictions, average='weighted')
    # Split conll_eval_lines into true and predicted sequences
    true_seqs = []
    pred_seqs = []
    for line in conll_eval_lines:
        if line.strip():  # Skip blank lines
            _, true_label, pred_label = line.split()
            true_seqs.append(true_label)
            pred_seqs.append(pred_label)
    # precision_, recall_, chunk_f1_ = evaluate_chunks(true_seqs, pred_seqs)

    # Evaluate using conlleval
    precision, recall, chunk_f1 = evaluate(true_seqs, pred_seqs)  # Unpack the tuple

    # Log chunk-level metrics
    logger.info(f"Chunk-level - Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {chunk_f1:.4f}")
    avg_loss = total_loss / len(data_loader)
    
    print("Avg_loss ::", avg_loss)
    print("Tag Level::", f1)
    print("Chunk F1 ::", chunk_f1/100)
    return avg_loss, f1, chunk_f1, all_targets, all_predictions

In [12]:


# Function to preprocess the data (BIO encoding)
def preprocess_data(file_path, output_path):
    logger.info(f"Preprocessing {file_path}...")
    preprocessed_data = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    for item in data:
        sentence = item['sentence']
        aspect_terms = item.get('aspect_terms', [])
        
        # Simple tokenization
        tokens = re.findall(r'\b\w+\b|[^\w\s]', sentence)
        
        # Initialize all labels as 'O'
        labels = ['O'] * len(tokens)
        
        # Extract aspect terms
        aspect_terms_list = []
        
        for aspect in aspect_terms:
            term = aspect['term']
            from_idx = aspect['from']
            to_idx = aspect['to']
            
            # Extract the exact term from the sentence
            aspect_term = sentence[int(from_idx):int(to_idx)]
            aspect_terms_list.append(aspect_term)
            
            # Find the tokens that correspond to this aspect term
            aspect_tokens = re.findall(r'\b\w+\b|[^\w\s]', aspect_term)
            
            # Look for these tokens in the original tokens list
            for i in range(len(tokens) - len(aspect_tokens) + 1):
                if labels[i] == 'I-TERM':
                    continue
                if [token.lower() for token in tokens[i:i+len(aspect_tokens)]] == [token.lower() for token in aspect_tokens]:
                    labels[i] = 'B-TERM'
                    for j in range(1, len(aspect_tokens)):
                        labels[i+j] = 'I-TERM'  # Inside
                    break
        
        # Create preprocessed example
        preprocessed_example = {
            'sentence': sentence,
            'tokens': tokens,
            'labels': labels,
            'aspect_terms': aspect_terms_list
        }
        
        preprocessed_data.append(preprocessed_example)
    
    # Save preprocessed data
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(preprocessed_data, f, ensure_ascii=False, indent=2)
    
    logger.info(f"Preprocessed data saved to {output_path}")
    return preprocessed_data

# Vocabulary and embedding matrix setup
class Vocabulary:
    def __init__(self, pad_token="<pad>", unk_token="<unk>"):
        self.token2idx = {pad_token: 0, unk_token: 1}
        self.idx2token = {0: pad_token, 1: unk_token}
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.pad_idx = 0
        self.unk_idx = 1
        
    def add_token(self, token):
        if token not in self.token2idx:
            self.token2idx[token] = len(self.token2idx)
            self.idx2token[len(self.idx2token)] = token
            
    def __len__(self):
        return len(self.token2idx)
    
    def token_to_idx(self, token):
        return self.token2idx.get(token, self.unk_idx)
    
    def idx_to_token(self, idx):
        return self.idx2token.get(idx, self.unk_token)
    
    def tokens_to_indices(self, tokens):
        return [self.token_to_idx(token) for token in tokens]
    
    def indices_to_tokens(self, indices):
        return [self.idx_to_token(idx) for idx in indices]

# Load embeddings
def evaluate_chunks(true_labels, pred_labels):
    def extract_chunks(labels):
        chunks = []
        current_chunk = None
        for i, label in enumerate(labels):
            if label.startswith('B'):
                if current_chunk:
                    chunks.append(tuple(current_chunk))
                current_chunk = [label[2:], i, i]  # (entity_type, start, end)
            elif label.startswith('I'):
                if current_chunk:
                    current_chunk[2] = i  # Extend the current chunk
            else:  # 'O' or other tags
                if current_chunk:
                    chunks.append(tuple(current_chunk))
                    current_chunk = None
        if current_chunk:
            chunks.append(tuple(current_chunk))
        return set(chunks)

    true_chunks = extract_chunks(true_labels)
    pred_chunks = extract_chunks(pred_labels)

    correct_chunks = true_chunks.intersection(pred_chunks)
    precision = len(correct_chunks) / len(pred_chunks) if pred_chunks else 0
    recall = len(correct_chunks) / len(true_chunks) if true_chunks else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1
    }
def load_glove_embeddings(vocab, embedding_dim=100):
    logger.info("Loading GloVe embeddings...")
    embeddings_dict = {}
    
    with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Loading GloVe"):
            values = line.split()
            word = values[0]
            vectors = np.asarray(values[1:], dtype='float32')
            embeddings_dict[word] = vectors
    
    embedding_matrix = np.random.uniform(-0.25, 0.25, (len(vocab), embedding_dim))
    embedding_matrix[0] = np.zeros(embedding_dim)  # Padding token
    
    # Initialize with pre-trained embeddings if available
    for word, idx in vocab.token2idx.items():
        if word in embeddings_dict:
            embedding_matrix[idx] = embeddings_dict[word]
    
    logger.info(f"GloVe embedding matrix created with shape: {embedding_matrix.shape}")
    return torch.FloatTensor(embedding_matrix)

def load_fasttext_embeddings(vocab, embedding_dim=300):
    logger.info("Loading FastText embeddings...")
    embeddings_dict = {}
    
    # Load first 100K vectors to save memory
    with open(FASTTEXT_PATH, 'r', encoding='utf-8') as f:
        next(f)  # Skip header
        for i, line in enumerate(tqdm(f, desc="Loading FastText")):
            if i >= 100000:  # Limit to first 100K vectors to save memory
                break
            values = line.split()
            word = values[0]
            vectors = np.asarray(values[1:], dtype='float32')
            embeddings_dict[word] = vectors
    
    embedding_matrix = np.random.uniform(-0.25, 0.25, (len(vocab), embedding_dim))
    embedding_matrix[0] = np.zeros(embedding_dim)  # Padding token
    
    # Initialize with pre-trained embeddings if available
    for word, idx in vocab.token2idx.items():
        if word in embeddings_dict:
            embedding_matrix[idx] = embeddings_dict[word]
    
    logger.info(f"FastText embedding matrix created with shape: {embedding_matrix.shape}")
    return torch.FloatTensor(embedding_matrix)

# Dataset and DataLoader
class AspectTermDataset(Dataset):
    def __init__(self, data, word_vocab, label_vocab):
        self.data = data
        self.word_vocab = word_vocab
        self.label_vocab = label_vocab
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        example = self.data[idx]
        tokens = example['tokens']
        labels = example['labels']
        
        # Convert tokens and labels to indices
        token_indices = self.word_vocab.tokens_to_indices(tokens)
        label_indices = self.label_vocab.tokens_to_indices(labels)
        
        return {
            'tokens': token_indices,
            'labels': label_indices,
            'length': len(token_indices)
        }

# Collate function for padding
def collate_fn(batch):
    tokens = [torch.tensor(item['tokens'][:MAX_LEN]) for item in batch]
    labels = [torch.tensor(item['labels'][:MAX_LEN]) for item in batch]
    lengths = [min(item['length'], MAX_LEN) for item in batch]
    
    # Pad sequences
    tokens_padded = pad_sequence(tokens, batch_first=True)
    labels_padded = pad_sequence(labels, batch_first=True)
    
    return {
        'tokens': tokens_padded,
        'labels': labels_padded,
        'lengths': torch.tensor(lengths)
    }

# RNN model
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix=None, bidirectional=True, dropout=0.3):
        super(RNNModel, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if embedding_matrix is not None:
            self.embedding.weight = nn.Parameter(embedding_matrix)
        
        # RNN layer
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Output layer
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        
    def forward(self, tokens, lengths):
        # Embedding
        embedded = self.embedding(tokens)
        
        # Pack sequence
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        # RNN
        output, hidden = self.rnn(packed)
        
        # Unpack sequence
        output, _ = pad_packed_sequence(output, batch_first=True)
        
        # Apply dropout
        output = self.dropout(output)
        
        # Output layer
        logits = self.fc(output)
        
        return logits

# GRU model
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix=None, bidirectional=True, dropout=0.3):
        super(GRUModel, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if embedding_matrix is not None:
            self.embedding.weight = nn.Parameter(embedding_matrix)
        
        # GRU layer
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Output layer
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        
    def forward(self, tokens, lengths):
        # Embedding
        embedded = self.embedding(tokens)
        
        # Pack sequence
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        # GRU
        output, hidden = self.gru(packed)
        
        # Unpack sequence
        output, _ = pad_packed_sequence(output, batch_first=True)
        
        # Apply dropout
        output = self.dropout(output)
        
        # Output layer
        logits = self.fc(output)
        
        return logits

# Training function
# Training function
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, model_name):
    logger.info(f"Starting training for {model_name}...")
    best_f1 = 0.0
    train_losses = []
    val_losses = []
    val_f1_scores = []
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            tokens = batch['tokens'].to(device)
            labels = batch['labels'].to(device)
            lengths = batch['lengths']
            
            # Forward pass
            optimizer.zero_grad()
            logits = model(tokens, lengths)
            
            # Reshape for loss calculation
            batch_size, max_len, num_classes = logits.shape
            logits_flat = logits.view(-1, num_classes)
            labels_flat = labels.view(-1)
            
            # Mask padding
            mask = (labels_flat != 0).float()
            
            # Compute loss
            loss = criterion(logits_flat, labels_flat) * mask
            loss = loss.sum() / mask.sum()
            
            # Backpropagation
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
        
        # Average loss for the epoch
        avg_train_loss = total_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation
        val_loss, tag_f1, chunk_f1, processed_targets, processed_predictions = evaluate_model(model, val_loader, criterion, label_vocab)
        chunk_f1 = chunk_f1/100
        val_losses.append(val_loss)
        val_f1_scores.append(chunk_f1)
        
        
        logger.info(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}, TAG_Val F1: {tag_f1:.4f}, Val Chunk F1: {chunk_f1:.4f}")
        
        # Save the best model
        if chunk_f1 > best_f1:
            best_f1 = chunk_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': val_loss,
                'Tag_val_f1': tag_f1,
                'val_chunk_f1': chunk_f1,
            }, f"best_model_{model_name}.pt")
            logger.info(f"Model saved with Chunk F1: {chunk_f1:.4f}")
    
    # Plot training/validation loss
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Training and Validation Loss for {model_name}')
    plt.legend()
    plt.savefig(f"{model_name}_loss_plot.png")
    plt.close()
    
    print(f"Returning: train_losses={train_losses}, val_losses={val_losses}, val_f1_scores={val_f1_scores}")
    return train_losses, val_losses, val_f1_scores

# Evaluation function
def predict(model, test_data, word_vocab, label_vocab):
    model.eval()
    all_predictions = []
    all_processed_sentences = []
    
    # Create dataset and dataloader
    test_dataset = AspectTermDataset(test_data, word_vocab, label_vocab)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=False)
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            tokens = batch['tokens'].to(device)
            lengths = batch['lengths']
            
            # Forward pass
            logits = model(tokens, lengths)
            
            # Get predictions
            _, predictions = torch.max(logits, dim=2)
            
            # Process predictions
            batch_size = tokens.size(0)
            for i in range(batch_size):
                length = lengths[i].item()
                pred = predictions[i, :length].cpu().tolist()
                token_indices = tokens[i, :length].cpu().tolist()
                
                # Convert indices to tokens and labels
                pred_labels = [label_vocab.idx_to_token(p) for p in pred]
                sentence_tokens = [word_vocab.idx_to_token(t) for t in token_indices]
                
                # Store predictions
                all_predictions.append(pred_labels)
                all_processed_sentences.append(sentence_tokens)
    
    return all_predictions, all_processed_sentences

# Function to extract aspect terms from predictions
def extract_aspect_terms(tokens, predictions):
    aspect_terms = []
    current_term = []
    
    for token, label in zip(tokens, predictions):
        if label.startswith('B'):
            if current_term:
                aspect_terms.append(' '.join(current_term))
                current_term = []
            current_term.append(token)
        elif label.startswith('I') and current_term:
            current_term.append(token)
    
    # Add the last term if there is one
    if current_term:
        aspect_terms.append(' '.join(current_term))
    
    return aspect_terms

# Function to load the best model and run inference on test data
def test_model(model_path, test_path, word_vocab, label_vocab):
    logger.info(f"Testing model from {model_path}...")
    
    # Load model architecture (either RNN or GRU)
    if 'RNN' in model_path:
        embedding_dim = EMBEDDING_DIM_GLOVE if 'GloVe' in model_path else EMBEDDING_DIM_FASTTEXT
        model = RNNModel(
            vocab_size=len(word_vocab), 
            embedding_dim=embedding_dim,
            hidden_dim=HIDDEN_DIM, 
            output_dim=len(label_vocab)
        ).to(device)
    else:
        embedding_dim = EMBEDDING_DIM_GLOVE if 'GloVe' in model_path else EMBEDDING_DIM_FASTTEXT
        model = GRUModel(
            vocab_size=len(word_vocab), 
            embedding_dim=embedding_dim,
            hidden_dim=HIDDEN_DIM, 
            output_dim=len(label_vocab)
        ).to(device)

    # Load model weights
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load and preprocess test data
    with open(test_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)

    preprocessed_test_data = []
    for item in test_data:
        sentence = item['sentence']
        tokens = re.findall(r'\b\w+\b|[^\w\s]', sentence)
        preprocessed_example = {
            'sentence': sentence,
            'tokens': tokens,
            'labels': ['O'] * len(tokens),  # Placeholder labels
            'aspect_terms': []  # Placeholder aspect terms
        }
        preprocessed_test_data.append(preprocessed_example)

    # Run inference
    predictions, processed_sentences = predict(model, preprocessed_test_data, word_vocab, label_vocab)

    # Extract aspect terms
    results = []
    conll_eval_lines = []  # To store lines for conlleval
    for i, (tokens, preds) in enumerate(zip(processed_sentences, predictions)):
        aspect_terms = extract_aspect_terms(tokens, preds)
        result = {
            'sentence': test_data[i]['sentence'],
            'tokens': tokens,
            'labels': preds,
            'aspect_terms': aspect_terms
        }
        results.append(result)

        # Create lines for conlleval (assuming all true labels are 'O')
        for token, pred_label in zip(tokens, preds):
            if token != word_vocab.pad_token:  # Skip padding tokens
                conll_eval_lines.append(f"{token} O {pred_label}")
        conll_eval_lines.append("")  # Blank line to separate sentences

    # Save results
    output_path = f"test_results_{os.path.basename(model_path).replace('.pt', '')}.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    logger.info(f"Test results saved to {output_path}")

    # Evaluate using conlleval
    precision, recall, chunk_f1 = evaluate(true_seqs, pred_seqs)  # Unpack the tuple

    logger.info(f"Test Results:")
    logger.info(f"Precision: {precision:.4f}")
    logger.info(f"Recall: {recall:.4f}")
    logger.info(f"F1: {chunk_f1:.4f}")

    return chunk_f1, results
# Main function
if __name__ == "__main__":
    # Preprocess data
    if not os.path.exists('train_task_1.json'):
        train_data = preprocess_data(TRAIN_PATH, 'train_task_1.json')
    else:
        with open('train_task_1.json', 'r', encoding='utf-8') as f:
            train_data = json.load(f)
            
    if not os.path.exists('val_task_1.json'):
        val_data = preprocess_data(VAL_PATH, 'val_task_1.json')
    else:
        with open('val_task_1.json', 'r', encoding='utf-8') as f:
            val_data = json.load(f)
    
    # Create vocabularies
    word_vocab = Vocabulary()
    label_vocab = Vocabulary(pad_token="O", unk_token="O")  # Default to 'O' tag
    
    # Add tokens to vocabularies
    for example in train_data:
        for token in example['tokens']:
            word_vocab.add_token(token)
    
    # Add labels
    for label in ['O', 'B-TERM', 'I-TERM']:
        label_vocab.add_token(label)
    
    logger.info(f"Word vocabulary size: {len(word_vocab)}")
    logger.info(f"Label vocabulary size: {len(label_vocab)}")
    
    # Create datasets
    train_dataset = AspectTermDataset(train_data, word_vocab, label_vocab)
    val_dataset = AspectTermDataset(val_data, word_vocab, label_vocab)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )
    
    # Save vocabularies
    with open('word_vocab.pkl', 'wb') as f:
        pickle.dump(word_vocab, f)
    
    with open('label_vocab.pkl', 'wb') as f:
        pickle.dump(label_vocab, f)
    
    # Load embeddings
    glove_embeddings = load_glove_embeddings(word_vocab, EMBEDDING_DIM_GLOVE)
    fasttext_embeddings = load_fasttext_embeddings(word_vocab, EMBEDDING_DIM_FASTTEXT)
    
    # Define models
    models = {
        'RNN-GloVe': RNNModel(
            vocab_size=len(word_vocab),
            embedding_dim=EMBEDDING_DIM_GLOVE,
            hidden_dim=HIDDEN_DIM,
            output_dim=len(label_vocab),
            embedding_matrix=glove_embeddings
        ).to(device),
        
        'RNN-FastText': RNNModel(
            vocab_size=len(word_vocab),
            embedding_dim=EMBEDDING_DIM_FASTTEXT,
            hidden_dim=HIDDEN_DIM,
            output_dim=len(label_vocab),
            embedding_matrix=fasttext_embeddings
        ).to(device),
        
        'GRU-GloVe': GRUModel(
            vocab_size=len(word_vocab),
            embedding_dim=EMBEDDING_DIM_GLOVE,
            hidden_dim=HIDDEN_DIM,
            output_dim=len(label_vocab),
            embedding_matrix=glove_embeddings
        ).to(device),
        
        'GRU-FastText': GRUModel(
            vocab_size=len(word_vocab),
            embedding_dim=EMBEDDING_DIM_FASTTEXT,
            hidden_dim=HIDDEN_DIM,
            output_dim=len(label_vocab),
            embedding_matrix=fasttext_embeddings
        ).to(device)
    }
    
    # Define loss function
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding index
    
    # Train and evaluate models
    results = {}
    for model_name, model in models.items():
        logger.info(f"Training {model_name}...")
        
        # Define optimizer
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        
        # Train model
        train_losses, val_losses, val_f1_scores = train_model(
            model,
            train_loader,
            val_loader,
            criterion,
            optimizer,
            EPOCHS,
            model_name
        )
        
        # Plot loss curves
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Training Loss')
        plt.plot(val_losses, label='Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_name} - Training and Validation Loss')
        plt.legend()
        plt.savefig(f'{model_name}_loss.png')
        plt.close()
        
        # Plot F1 scores
        plt.figure(figsize=(10, 6))
        plt.plot(val_f1_scores, label='Validation F1 Score')
        plt.xlabel('Epoch')
        plt.ylabel('F1 Score')
        plt.title(f'{model_name} - Validation F1 Score')
        plt.legend()
        plt.savefig(f'{model_name}_f1.png')
        plt.close()
        
        # Save model
        torch.save(model.state_dict(), f'{model_name}.pt')
        
        # Save results
        results[model_name] = {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_f1_scores': val_f1_scores,
            'best_f1': max(val_f1_scores)
        }
    
    # Compare models
    logger.info("Model comparison:")
    for model_name, result in results.items():
        logger.info(f"{model_name}: Best F1 Score = {result['best_f1']:.4f}")
    
    # Find best model
    best_model_name = max(results, key=lambda x: results[x]['best_f1'])
    logger.info(f"Best model: {best_model_name} with F1 Score = {results[best_model_name]['best_f1']:.4f}")
    # Inside your main function, after training all models, add these lines:

    # Already have results dictionary with model data
    
    # Create a more detailed comparison for reporting
    logger.info("\n===== DETAILED MODEL COMPARISON =====")
    logger.info(f"{'Model':<15} {'Best F1':<10} {'Best Epoch':<12} {'Final Training Loss':<20} {'Final Validation Loss':<20}")
    for model_name, result in results.items():
        best_epoch = result['val_f1_scores'].index(result['best_f1']) + 1
        logger.info(f"{model_name:<15} {result['best_f1']:.4f}     {best_epoch:<12} {result['train_losses'][-1]:.6f}         {result['val_losses'][-1]:.6f}")
    
    # Create a bar chart comparison of F1 scores
    plt.figure(figsize=(10, 6))
    model_names = list(results.keys())
    f1_scores = [results[model]['best_f1'] for model in model_names]
    plt.bar(model_names, f1_scores)
    plt.xlabel('Model')
    plt.ylabel('Best F1 Score')
    plt.title('Model Comparison - Best F1 Scores')
    plt.savefig('model_comparison_f1.png')
    plt.close()
    
    # Save the comparison data to a file for easy inclusion in report
    with open('model_comparison_results.txt', 'w') as f:
        f.write("Model Comparison Results\n")
        f.write("======================\n\n")
        f.write(f"{'Model':<15} {'Best F1':<10} {'Best Epoch':<12} {'Final Training Loss':<20} {'Final Validation Loss':<20}\n")
        for model_name, result in results.items():
            best_epoch = result['val_f1_scores'].index(result['best_f1']) + 1
            f.write(f"{model_name:<15} {result['best_f1']:.4f}     {best_epoch:<12} {result['train_losses'][-1]:.6f}         {result['val_losses'][-1]:.6f}\n")
        
        f.write(f"\nBest Model: {best_model_name} with F1 Score = {results[best_model_name]['best_f1']:.4f}")
    
    logger.info(f"\nComparison chart saved to model_comparison_f1.png")
    logger.info(f"Detailed results saved to model_comparison_results.txt")

def evaluate_test(test_file, model_path, model_type='GRU', embedding_type='GloVe'):
    """
    Function to evaluate the model on test data
    
    Args:
        test_file: Path to the test file
        model_path: Path to the saved model
        model_type: 'RNN' or 'GRU'
        embedding_type: 'GloVe' or 'FastText'
    
    Returns:
        F1 scores (tag-level and chunk-level)
    """
    # Load vocabularies
    with open('word_vocab.pkl', 'rb') as f:
        word_vocab = pickle.load(f)
    
    with open('label_vocab.pkl', 'rb') as f:
        label_vocab = pickle.load(f)
    
    # Preprocess test data
    test_data = preprocess_data(test_file, 'test_task_1.json')
    
    # Create test dataset
    test_dataset = AspectTermDataset(test_data, word_vocab, label_vocab)
    
    # Create test dataloader
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )
    
    # Load model
    if model_type == 'RNN':
        if embedding_type == 'GloVe':
            model = RNNModel(
                vocab_size=len(word_vocab),
                embedding_dim=EMBEDDING_DIM_GLOVE,
                hidden_dim=HIDDEN_DIM,
                output_dim=len(label_vocab)
            ).to(device)
        else:  # FastText
            model = RNNModel(
                vocab_size=len(word_vocab),
                embedding_dim=EMBEDDING_DIM_FASTTEXT,
                hidden_dim=HIDDEN_DIM,
                output_dim=len(label_vocab)
            ).to(device)
    else:  # GRU
        if embedding_type == 'GloVe':
            model = GRUModel(
                vocab_size=len(word_vocab),
                embedding_dim=EMBEDDING_DIM_GLOVE,
                hidden_dim=HIDDEN_DIM,
                output_dim=len(label_vocab)
            ).to(device)
        else:  # FastText
            model = GRUModel(
                vocab_size=len(word_vocab),
                embedding_dim=EMBEDDING_DIM_FASTTEXT,
                hidden_dim=HIDDEN_DIM,
                output_dim=len(label_vocab)
            ).to(device)
    
    # Load model weights
    model.load_state_dict(torch.load(model_path))
    
    # Evaluate model
    model.eval()
    all_true_labels = []
    all_pred_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            tokens = batch['tokens'].to(device)
            labels = batch['labels'].to(device)
            lengths = batch['lengths']
            
            # Forward pass
            logits = model(tokens, lengths)
            
            # Get predictions
            predictions = torch.argmax(logits, dim=2)
            
            # Convert to list
            for i, length in enumerate(lengths):
                true_labels = labels[i, :length].cpu().tolist()
                pred_labels = predictions[i, :length].cpu().tolist()
                
                # Convert indices to tokens
                true_labels = label_vocab.indices_to_tokens(true_labels)
                pred_labels = label_vocab.indices_to_tokens(pred_labels)
                
                all_true_labels.extend(true_labels)
                all_pred_labels.extend(pred_labels)
    
    # Evaluate using conlleval
    true_chunks, pred_chunks = [], []
    for true_label, pred_label in zip(all_true_labels, all_pred_labels):
        if true_label != label_vocab.pad_token:  # Skip padding tokens
            true_chunks.append(true_label)
            pred_chunks.append(pred_label)
    
    # Format for evaluation
    precision, recall, chunk_f1 = evaluate(true_seqs, pred_seqs)  # Unpack the tuple

    logger.info(f"Test Results:")
    logger.info(f"  Precision: {precision:.4f}")
    logger.info(f"  Recall: {recall:.4f}")
    logger.info(f"  F1: {chunk_f1:.4f}")
    
    logger.info("Test Results:")
    logger.info(f"Chunk-level - Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {chunk_f1:.4f}")
    
    return eval_result

Loading GloVe: 400000it [00:08, 46154.27it/s]
Loading FastText: 100000it [00:05, 16799.78it/s]
Evaluating: 100%|██████████| 10/10 [00:00<00:00, 160.75it/s]


processed 5045 tokens with 120 phrases; found: 64 phrases; correct: 41.
accuracy:  34.17%; (non-O)
accuracy:  97.98%; precision:  64.06%; recall:  34.17%; FB1:  44.57
             TERM: precision:  64.06%; recall:  34.17%; FB1:  44.57  64
Avg_loss :: 0.0666893869638443
Tag Level:: 0.9767624521277793
Chunk F1 :: 0.44565217391304346


Evaluating: 100%|██████████| 10/10 [00:00<00:00, 152.37it/s]


processed 5045 tokens with 120 phrases; found: 62 phrases; correct: 48.
accuracy:  40.00%; (non-O)
accuracy:  98.30%; precision:  77.42%; recall:  40.00%; FB1:  52.75
             TERM: precision:  77.42%; recall:  40.00%; FB1:  52.75  62
Avg_loss :: 0.05014001354575157
Tag Level:: 0.9802870999171314
Chunk F1 :: 0.5274725274725275


Evaluating: 100%|██████████| 10/10 [00:00<00:00, 159.85it/s]


...

processed 5045 tokens with 120 phrases; found: 149 phrases; correct: 83.
accuracy:  69.17%; (non-O)
accuracy:  97.96%; precision:  55.70%; recall:  69.17%; FB1:  61.71
             TERM: precision:  55.70%; recall:  69.17%; FB1:  61.71  149
Avg_loss :: 0.07204530593007803
Tag Level:: 0.980654107536025
Chunk F1 :: 0.6171003717472119


Evaluating: 100%|██████████| 10/10 [00:00<00:00, 164.57it/s]


processed 5045 tokens with 120 phrases; found: 137 phrases; correct: 76.
accuracy:  63.33%; (non-O)
accuracy:  97.92%; precision:  55.47%; recall:  63.33%; FB1:  59.14
             TERM: precision:  55.47%; recall:  63.33%; FB1:  59.14  137
Avg_loss :: 0.07274565063416957
Tag Level:: 0.9798576802492058
Chunk F1 :: 0.5914396887159533


Evaluating: 100%|██████████| 10/10 [00:00<00:00, 165.01it/s]


processed 5045 tokens with 120 phrases; found: 130 phrases; correct: 78.
accuracy:  65.00%; (non-O)
accuracy:  98.14%; precision:  60.00%; recall:  65.00%; FB1:  62.40
             TERM: precision:  60.00%; recall:  65.00%; FB1:  62.40  130
Avg_loss :: 0.07556050531566143
Tag Level:: 0.9817308693304971
Chunk F1 :: 0.624
Returning: train_losses=[0.20383340246104575, 0.06003451473140097, 0.03342087432748166, 0.02053222239211008, 0.01322528334975533, 0.009326775492620261, 0.005938000941840711, 0.004086973649724054, 0.003288095545775088, 0.0026538105560209627], val_losses=[0.0660393126308918, 0.047303926665335894, 0.05199360586702824, 0.04946633521467447, 0.06184741966426373, 0.06062904326245189, 0.06505275107920169, 0.07204530593007803, 0.07274565063416957, 0.07556050531566143], val_f1_scores=[0.378698224852071, 0.5786802030456852, 0.6332046332046332, 0.6422764227642277, 0.6014492753623187, 0.6351931330472103, 0.6299212598425197, 0.6171003717472119, 0.5914396887159533, 0.624]
